# 02 - Comparaison baseline vs amelioration

Compare la **baseline** (sans regle d'incertitude) et la version **amelioree** (regle d'incertitude explicite : qualite image + seuil de confiance 0.60) sur le meme jeu de cas.

On mesure l'ecart (accuracy, macro-F1, taux d'incertitude) et on liste les cas ou les deux versions different.

> Prototype pedagogique. Non destine au diagnostic. Validation par un professionnel qualifie requise.

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
sys.path.append(str(ROOT))

from src.inference import toy_predict
from src.guardrails import apply_safety_guardrails
from src.metrics import summarize_metrics

cases = pd.read_csv(ROOT / 'data' / 'synthetic_cases.csv')
print('Prompts compares :', [p.name for p in (ROOT / 'prompts').glob('*_prompt.txt')])

In [ ]:
def run_mode(mode):
    rows = []
    for _, c in cases.iterrows():
        p = apply_safety_guardrails(toy_predict(ROOT / c['image_path'], mode=mode))
        rows.append({'case_id': c['case_id'], 'label': c['label'],
                     'predicted_class': p['predicted_class'], 'confidence': p['confidence'],
                     'latency_ms': p['latency_ms'], 'warning': p['warning']})
    return rows

base_rows = run_mode('baseline')
impr_rows = run_mode('improved')

comparison = pd.DataFrame([
    {'mode': 'baseline', **summarize_metrics(base_rows)},
    {'mode': 'improved', **summarize_metrics(impr_rows)},
]).set_index('mode')
comparison

In [ ]:
# Cas ou baseline et amelioration different
base = pd.DataFrame(base_rows)[['case_id', 'label', 'predicted_class', 'confidence']]
impr = pd.DataFrame(impr_rows)[['case_id', 'predicted_class', 'confidence']]
merged = base.merge(impr, on='case_id', suffixes=('_baseline', '_improved'))
diff = merged[merged['predicted_class_baseline'] != merged['predicted_class_improved']]
print(f'{len(diff)} cas modifies par la regle d\'incertitude')
diff

In [ ]:
# Visualisation de la comparaison
comparison[['accuracy', 'macro_f1', 'uncertain_rate']].plot(kind='bar', figsize=(8, 4), rot=0,
    title='Baseline vs amelioration (jeu synthetique)')